# Lab 4: an answer service that can prove where its answer came from

**Time: about 80 minutes. Work in pairs.** One person types, the other checks.

**What you will build.** A small service that takes a question, retrieves with the Lab 3 stack,
writes an answer, and returns that answer together with the exact pieces of text it used. You will
then break it on purpose, by making it cite a piece of text that was never retrieved, and watch the
check catch it. Last, you will try to make it refuse the five questions the documents cannot answer,
and find out that no single setting does it cleanly.

**How the cells work.** Each step has up to three cells: look at the real thing, make your call, then
see the measurement. Every `YOUR CALL` cell holds a working value, so the notebook runs top to bottom
before you edit anything.

**Words used throughout.** *Grounding* means every sentence in the answer comes from retrieved text
rather than from the model's memory. A *citation* is the identifier of the exact piece of text used, so
a reader can go and check it.

**The scores, by plain name.** The industry term is in brackets, because tools such as Ragas and MLflow
use it.

| Plain name | Industry term | What it checks |
|---|---|---|
| Match score | retrieval score; the response field is `confidence` | how well the best retrieved piece's words match the question, from 0 to 1. It is not how sure the model is |
| Supported sentences | faithfulness | the share of the answer's sentences whose words appear in the retrieved text, from 0 to 1 |
| Right fact | correct facts: the way Lab 4 measures answer relevance | the answer states the fact the question asked for |
| Real citations | citation check | every identifier the answer cites was actually retrieved |
| Cites only what it used | citation precision | of the pieces an answer cites, the share it actually used (Step 6) |
| Every claim cited | citation recall | of the sentences that state something, the share that carry a citation to a piece that supports them (Step 6) |

In [ ]:
%pip install -q pypdf python-docx python-pptx openpyxl rank_bm25

In [ ]:
# Setup for Databricks. Run this cell first: every later cell depends on it.
# It puts the lab files (src/ and data/) in this session's own folder, /tmp/freshcart, and puts
# src/ on the import path so `from freshcart import ...` works.
# If you opened this notebook from a Git folder of the repository, the files are copied from that
# folder. If you imported only this notebook, they are downloaded from GitHub.
import io, os, sys, pathlib, shutil, textwrap, urllib.request, zipfile

REPO_ZIP = "https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip"
ROOT = pathlib.Path("/tmp/freshcart")


def repo_folder():
    here = pathlib.Path(os.getcwd())          # on Databricks, the folder that holds this notebook
    for p in (here, *here.parents):
        if (p / "src" / "freshcart.py").is_file() and (p / "data" / "raw").is_dir():
            return p
    return None


ROOT.mkdir(parents=True, exist_ok=True)
found = repo_folder()
if found:
    for part in ("src", "data"):
        shutil.copytree(found / part, ROOT / part, dirs_exist_ok=True, copy_function=shutil.copyfile)
    print("lab files copied from your Git folder:", found)
else:
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(REPO_ZIP, timeout=120).read())) as z:
        for name in z.namelist():
            rel = name.split("/", 1)[1] if "/" in name else ""
            if rel.startswith(("src/", "data/")) and not name.endswith("/"):
                (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
                (ROOT / rel).write_bytes(z.read(name))
    print("lab files downloaded from GitHub")
for folder in ("out", "data/cache", "data/logs"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

## The answer writer: a real language model

**What changes in this notebook.** The answer writer is `databricks-meta-llama-3-3-70b-instruct`, a
language model this Databricks workspace serves. It reads the grounding prompt, the five retrieved
pieces of text and the question, and writes the answer in its own words, citing each piece it used.
Retrieval is unchanged: it is the Lab 3 stack, so the scores in Step 4 are the ones you already know.

**What you need to know before you run it.**

- **No keys.** The notebook signs in to the model as you. Each answer costs a fraction of a cent,
  billed to your workspace.
- **Each question is answered once.** The writer remembers its answers, so Steps 2, 3 and 5 reuse
  them instead of asking the model again. About 25 model calls per run.
- **If the model refuses a call**, most likely because many people are running at once, the writer
  waits and tries again. If it still cannot get an answer, it falls back to the offline writer for
  that one question and says so in the output.

**What you should see.** One line naming the model, then `the model answered: this run is live`.

In [ ]:
import re, time, threading
from databricks.sdk import WorkspaceClient

WRITER_MODEL = "databricks-meta-llama-3-3-70b-instruct"
_ws = WorkspaceClient()                 # signs in as whoever runs the notebook: no key
_memo, _lock = {}, threading.Lock()
FALLBACKS = []                          # questions answered by the offline writer instead

def ask_model(messages, max_tokens=350):
    """One call to the served model. The built-in models allow only a few requests a
    second for the whole workspace, so a refused call waits and tries again."""
    for attempt in range(6):
        try:
            r = _ws.api_client.do("POST", f"/serving-endpoints/{WRITER_MODEL}/invocations",
                                  body={"messages": messages, "temperature": 0, "max_tokens": max_tokens})
            return r["choices"][0]["message"]["content"] or ""
        except Exception as e:
            if attempt == 5:
                raise
            time.sleep(2 * (attempt + 1))

def model_writer(prompt, context, question=None):
    """The answer writer. Same inputs as the offline one: the grounding prompt, the
    retrieved pieces each tagged with its id, and the question."""
    key = (question, context)
    with _lock:
        if key in _memo:
            return _memo[key]
    messages = [{"role": "system", "content": prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
    try:
        text = ask_model(messages).strip()
    except Exception as e:
        from rag import echo_generator
        FALLBACKS.append(question)
        print(f"  model unavailable for one question ({type(e).__name__}); used the offline writer")
        text = echo_generator(prompt, context, question)
    with _lock:
        _memo[key] = text
    return text

print("answer writer:", WRITER_MODEL)
ask_model([{"role": "user", "content": "Reply with the word ok."}], max_tokens=5)
print("the model answered: this run is live")

## Step 1. Look at the whole response, not just the answer

**What you will do.** Run the next cell. It builds the service and asks one real question, then prints
the complete response as the service would return it over the network.

**What you should see.** Six fields: `answer`, `citations`, `confidence`, `retrieved`, `refused` and
`refusal_reason`. The answer is a sentence the model wrote, "A customer may make up to three refund
claims in any rolling thirty day period", ending with a short identifier in square brackets.
`citations` holds that same identifier. `retrieved` holds five identifiers, because retrieval returned
five pieces and the answer used one of them. `refused` is false. `confidence` is the match score of the best piece, 0.749 here: how well
its words match the question, not how sure anyone is of the answer.

**Why this step is here.** Most teams return a string and nothing else. Every hard question that comes
later, such as "where did that come from" and "why did it say that", is unanswerable without the other
five fields. The shape of this response is the deliverable of the lab, more than the answer text.

In [ ]:
# RagStack is the whole answer service in one object: it holds the Lab 3 retrieval
# stack, the grounding prompt, the answer writer and the citation check.
# faithfulness gives the supported-sentences score: the share of the answer's sentences
# whose words appear in the retrieved text.
import json, statistics
from ragstack import RagStack
from rag import faithfulness, GROUNDING_PROMPT
from labkit import load_golden, table, banner

svc = RagStack(False, threshold=0.15)    # threshold: refuse when the best score is below this
svc.generate = model_writer               # the real language model writes the answers
ans, hits = svc.ask("How many refund claims can a customer make in thirty days?")
print(json.dumps(ans.to_dict(), indent=1)[:900])
print("\nretrieved pieces, best first:")
for score, ch in hits:
    print(f"  {score:5.3f}  {ch.id}  {ch.meta['source']}")

## Step 2. Break it on purpose: a citation for text that was never retrieved

**What you will do.** Wrap the answer writer so that every third answer has an extra identifier added
to it, one that was never retrieved. Then run 25 questions and count how many the citation check
catches.

**What you should see.** 8 answers carry the fake identifier and all 8 are caught and refused, with
the reason starting `unverifiable citations`. The service refuses rather than returning a sentence
that points at nothing.

**Why this step is here.** A model that invents a source is the failure that ends up in the newspaper,
and the Air Canada case study on this day is exactly that. You cannot test for it by reading answers,
because invented citations look like real ones. You test for it by checking, in code, that every
identifier in the answer is one that retrieval actually returned.

In [ ]:
# The golden set is deliberately in two groups. Refusing an unanswerable question is
# right; refusing an answerable one is a failure. A single refusal rate hides both.
gold = load_golden("qa_golden.json")
questions = [(g, item) for g in ("answerable", "unanswerable") for item in gold[g]]
print(len(gold["answerable"]), "questions the documents can answer")
print(len(gold["unanswerable"]), "questions they cannot")

In [ ]:
# YOUR CALL: how many of the 25 answers do you think will carry a fake citation,
# and how many of those will be caught?
PREDICT_FAKES = 8
PREDICT_CAUGHT = 8

In [ ]:
# Fault injection: wrap the answer writer so every third answer carries an identifier
# that retrieval never returned. This is what an invented source looks like in code.
def phantom_every_third(generate):
    state = {"n": 0}
    def wrapped(prompt, context, question=None):
        text = generate(prompt, context, question)
        state["n"] += 1
        return text + " [deadbeef0000]" if state["n"] % 3 == 0 else text
    return wrapped

broken = RagStack(False, threshold=0.15)
broken.generate = phantom_every_third(model_writer)
caught = 0
for group, item in questions:
    a, h = broken.ask(item["question"])
    if (a.refusal_reason or "").startswith("unverifiable citations"):
        caught += 1
print(f"answers carrying a fake citation, caught and refused: {caught}")
print(f"you predicted {PREDICT_CAUGHT} caught out of {PREDICT_FAKES} faked")
print("\nnote what did NOT happen: nothing raised an error, and no answer text looked wrong.")

### How the fake citation was made, and how it was caught

**The model was never asked to lie.** The cell above puts a small piece of code, a *wrapper*, between the
model and the service. A wrapper is a function that calls the real answer writer and then changes its
output before passing it on. This one counts answers and, on every third one, adds ` [deadbeef0000]` to
the end. That identifier has the same shape as a real one such as `37801390e45c`, so a reader cannot tell
it apart. 25 answers give 8 faked ones: numbers 3, 6, 9, 12, 15, 18, 21 and 24.

A faked answer reads like this:

> A customer may make up to three refund claims in any rolling thirty day period [37801390e45c]. [deadbeef0000]

**How the service catches it.** The citation check in `src/rag.py` runs on every answer:

```python
cited   = re.findall(r"\[([a-f0-9]{6,16})\]", text)   # every [id] in the answer
valid   = [c for c in cited if c in retrieved_ids]   # ids retrieval really returned
phantom = [c for c in cited if c not in retrieved_ids]
if phantom:                                          # one bad id is enough
    refuse, with the reason "unverifiable citations: ['deadbeef0000']"
```

Retrieval returned 5 identifiers for the question, and `deadbeef0000` is not one of them, so the whole
answer is refused, even though its real sentence was correct. The cell then counts the refusals whose
reason starts with `unverifiable citations`: 8 caught out of 8 faked.

**Why fake it on purpose.** A real model invents a source only now and then, so a 25-question test
cannot count on seeing one. Putting in a known fault on a fixed schedule tests the check itself: we know
exactly how many fakes went in, so we can prove every one came out. This is called *fault injection*.
It costs no extra model calls, because the writer reuses the answers it saved in Step 1.


## Step 3. Measure the honest service on all 25 questions

**What you will do.** Run the real service over the 20 answerable and 5 unanswerable questions at the
default setting, and print the four numbers that matter.

**What you should see.** Close to these, measured on 8 October:

| Measure | Result | Want |
|---|---|---|
| refused, answerable | 1 of 20 | low |
| refused, unanswerable | 4 of 5 | high |
| supported sentences (faithfulness) | 1.000 | near 1 |
| right fact (correct facts) | 19 of 20 | high |

The model refuses most unanswerable questions by itself: it writes that it could not find the answer
and cites nothing, and the service turns "no citation" into a refusal. Your numbers can differ a little
from a classmate's, because a language model does not always word an answer the same way.

**Why this step is here.** A real answer writer fixes most of what the offline one got wrong. It does not
fix everything, and the two answers it gets wrong are the most useful ones in the lab. Step 5 shows them.

In [ ]:
# Ask all 25 questions and record four things per answer: was it refused, the match score
# of the best piece, the share of supported sentences, and does it state the expected fact.
rows = []
for group, item in questions:
    a, h = svc.ask(item["question"])
    text = a.answer or ""
    rows.append({"id": item["id"], "group": group, "refused": a.refused,
                 "match score": round(h[0][0], 3) if h else 0.0,
                 "supported": round(faithfulness(text, h), 3) if text else None,
                 "right fact": (any(e.lower() in text.lower() for e in item.get("expect_any", []))
                             if group == "answerable" else None)})
print(table(rows, ["id", "group", "refused", "match score", "supported", "right fact"]))

ansb = [r for r in rows if r["group"] == "answerable"]
unans = [r for r in rows if r["group"] == "unanswerable"]
banner("the four numbers")
print(f"  refused, answerable   : {sum(r['refused'] for r in ansb)}/{len(ansb)}   (want LOW)")
print(f"  refused, unanswerable : {sum(r['refused'] for r in unans)}/{len(unans)}   (want HIGH)")
print(f"  supported sentences   : {statistics.mean(r['supported'] for r in ansb if r['supported']):.3f}   (faithfulness)")
print(f"  right fact            : {sum(bool(r['right fact']) for r in ansb)}/{len(ansb)}   (correct facts)")

## Step 4. Could a match-score threshold do the refusing instead?

**What you will do.** The service can also refuse before the model is asked, when the best match
score is below a threshold. In the `YOUR CALL` cell, choose the threshold you would ship. The reveal
prints the two score ranges and sweeps the threshold so you can see what each choice costs.

**What you should see.** The 20 answerable questions score between 0.641 and 0.915. The 5 unanswerable
ones score 0.468, 0.562, 0.581, 0.620 and 0.740. The two ranges overlap, so the sweep shows 0.6
refusing 3 of 5 unanswerable and none of the answerable, while 0.7 catches 4 of 5 but starts refusing
3 real questions.

**Why this step is here.** A match score says how similar the text is, not whether it answers the
question, so no threshold separates the two groups cleanly. That is why the model's own "I could not
find it", checked in code, does better in Step 3 than any threshold here. The two work together: a
threshold catches the clearly hopeless questions cheaply, before any model call is paid for.

In [ ]:
# YOUR CALL: the refusal threshold you would ship
THRESHOLD = 0.6

In [ ]:
# Put the two groups of match scores side by side. If they do not separate cleanly,
# no threshold can separate the questions either, whatever number you pick.
a_scores = sorted(r["match score"] for r in ansb)
u_scores = sorted(r["match score"] for r in unans)
print(f"answerable  : min {a_scores[0]:.3f}  median {a_scores[len(a_scores)//2]:.3f}  max {a_scores[-1]:.3f}")
print(f"unanswerable: {u_scores}")
print("\nthe ranges overlap, so no single cut separates them:\n")
sweep = []
for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    sweep.append({"threshold": t,
                  "refused answerable": f"{sum(1 for s in a_scores if s < t)}/20",
                  "refused unanswerable": f"{sum(1 for s in u_scores if s < t)}/5"})
print(table(sweep, ["threshold", "refused answerable", "refused unanswerable"]))
print(f"\nyour choice was {THRESHOLD}: "
      f"{sum(1 for s in a_scores if s < THRESHOLD)}/20 answerable refused, "
      f"{sum(1 for s in u_scores if s < THRESHOLD)}/5 unanswerable refused")

## Step 5. The two answers the code got wrong

**What you will do.** Run the next cell. It prints every answer where what the model wrote and what the
service decided disagree.

**What you should see.** Two of them.

- **A15**, "What is the first response TAT for a food safety complaint?" The model answered correctly,
  15 minutes, but put more than an identifier inside the square brackets, for example
  `[88ddf531a0f2, 934022623e36]` or `[88ddf531a0f2, table 1]`. The citation check looks for an
  identifier on its own in square brackets, found none, and refused a correct answer.
- **U01**, "What is FreshCart's refund policy for grocery orders in Singapore?" The model wrote that it
  could not find the answer, and then cited a piece of text to explain why. Because a valid citation is
  present, the service counted this as an answer, not a refusal. The words say no; the fields say yes.

**Why this step is here.** With a real model, the hard part moves from the model to the code around it.
Both failures are format problems: the model did the right thing and the service read it wrongly. The
fixes are in code, not in the model: read every identifier inside a bracket, not just a lone one, and treat "could not find"
as a refusal whatever else the answer contains. Take one of the two to the debrief and say how you would
fix it.

In [ ]:
# Where the model and the service disagree: a refused answer that states the expected fact,
# or an accepted answer whose words say the model could not find it.
said_no = ("could not find", "cannot find", "can't find", "not mentioned", "does not mention", "no information")
cases = []
for group, item in questions:
    a, h = svc.ask(item["question"])                     # answers are remembered: no new model calls
    context = "\n\n".join(f"[{c.id}] {c.text}" for _, c in h)
    raw = model_writer(GROUNDING_PROMPT, context, item["question"])
    hit = any(e.lower() in raw.lower() for e in item.get("expect_any", []))
    if a.refused and group == "answerable" and hit:
        cases.append((item, raw, f"refused ({a.refusal_reason}), but the answer was right"))
    if not a.refused and any(p in raw.lower() for p in said_no):
        cases.append((item, raw, "counted as answered, but the model said it could not find it"))
for item, raw, why in cases:
    print(f"{item['id']}  Q: {item['question']}")
    print(f"      model wrote: {raw[:200]}")
    print(f"      service    : {why}\n")
print(len(cases), "answers where the code read the model wrongly")

## Step 6. Two citation checks the service does not make yet

**What you will do.** Run the next cell. It reads the answers already stored, so it makes no new model
calls, and scores every answer the service accepted on two more checks.

- **Cites only what it used (citation precision).** Of the pieces an answer cites, the share it actually
  used. A cited piece counts as used when at least one sentence of the answer is supported by that piece
  alone: 60% or more of the sentence's words of 4 or more letters appear in it. Low precision means the
  model adds extra citations to look safe.
- **Every claim cited (citation recall).** Of the sentences that state something, the share that carry a
  citation to a piece that supports them. A sentence that says the model could not find the answer is not
  a claim and is left out. Low recall means facts with no proof of where they came from.

**What you should see.** Close to this, measured on 9 October. Cites only what it used: 20 of 22 cited
pieces were used, 0.91. Every claim cited: 19 of 19 claims carry a supporting citation, 1.00. Two answers
lose points:

- **A16**, "Who bears the refund cost when the failure is caused by delivery partner handling?" The model
  cites two pieces of the partner agreement. The second holds the end of the same clause, which chunking cut
  in two, and only 54% of the sentence's words are in it, so the rule calls it unused. Read it and decide:
  this citation is arguably fair, and the number is a word check, not a judge.
- **U01** cites a piece while saying it could not find the answer. It makes no claim, so the citation
  supports nothing.

Recall is perfect here because the model follows the prompt's "cite every factual claim" and writes
one-sentence answers. Longer answers are where recall drops. Your numbers can differ a little, because the
model does not always word an answer the same way.

**Why this step is here.** The citation check in Step 2 only asks whether each cited identifier was
retrieved. It does not ask whether the piece supports the sentence it is attached to, or whether every
sentence has one. These two numbers do. They are stricter than supported sentences (faithfulness), which
compares each sentence with all 5 retrieved pieces together, not with the piece it cites.

In [ ]:
# Citation precision and recall, from the stored answers: no new model calls.
# A piece "supports" a sentence when 60% or more of the sentence's words of 4+ letters appear in it,
# the same rule as supported sentences, but checked against one cited piece instead of all five.
ID = re.compile(r"[a-f0-9]{12}")
def words(sentence):
    return re.findall(r"[a-z]{4,}", re.sub(r"\[[^\]]*\]", " ", sentence).lower())
def supports(sentence, piece_text):
    w = words(sentence)
    return bool(w) and sum(x in piece_text.lower() for x in w) / len(w) >= 0.6

cite_rows, cited_all, used_all, claims_all, backed_all = [], 0, 0, 0, 0
for group, item in questions:
    a, h = svc.ask(item["question"])                     # answers are remembered: no new model calls
    if a.refused:
        continue
    pieces = {c.id: c.text for _, c in h}
    sents = [s for s in re.split(r"(?<=[.!?])\s+", a.answer) if s.strip()]
    cited = [c for c in dict.fromkeys(ID.findall(a.answer)) if c in pieces]
    used = [c for c in cited if any(supports(s, pieces[c]) for s in sents)]
    claims = [s for s in sents if words(s) and not any(p in s.lower() for p in said_no)]
    backed = [s for s in claims if any(supports(s, pieces[c]) for c in ID.findall(s) if c in pieces)]
    cited_all += len(cited); used_all += len(used); claims_all += len(claims); backed_all += len(backed)
    if len(used) < len(cited) or len(backed) < len(claims):
        cite_rows.append({"id": item["id"], "cited": len(cited), "used": len(used),
                          "claims": len(claims), "cited and supported": len(backed)})

print(f"answers accepted: {sum(1 for g, i in questions if not svc.ask(i['question'])[0].refused)}")
print(f"cites only what it used (citation precision): {used_all}/{cited_all} cited pieces were used"
      f" = {used_all / max(cited_all, 1):.2f}")
print(f"every claim cited (citation recall)          : {backed_all}/{claims_all} claims carry a supporting citation"
      f" = {backed_all / max(claims_all, 1):.2f}")
if cite_rows:
    print("\nanswers that lose points:")
    print(table(cite_rows, ["id", "cited", "used", "claims", "cited and supported"]))
    for r in cite_rows:
        item = next(i for g, i in questions if i["id"] == r["id"])
        print(f"\n{r['id']}  {svc.ask(item['question'])[0].answer[:300]}")


## When you are done

- Fill in the **Your write-up** cell at the bottom of this notebook: the four numbers, your threshold and what it costs, which
  of the two answers the code read wrongly you would fix first, and the two citation numbers from Step 6.
- Run it as a real service: `python labs\lab04_rag_api.py serve`, then open
  `http://127.0.0.1:8000/docs` and ask it a question from the browser.
- Run it as a service on your own computer: see **Lab 4 as an API** in the README.

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it, and that export is what you
hand in.

Do not write in `handouts/lab4_answers.md`. It is the blank answer sheet, and a fresh copy of the repository replaces it.

---

**Pair:**

### 1. The response contract

List the six fields your service returns, and for each one, who needs it and why.

| Field | Who needs it | Why |
|---|---|---|
| | | |

### 2. The four numbers, reported separately

| Measure | Value | Should be |
|---|---|---|
| Refusal rate, answerable questions | /20 | low |
| Refusal rate, unanswerable questions | /5 | high |
| Supported sentences (faithfulness), answered questions | | near 1 |
| Right fact (correct facts), answerable questions | /20 | high |

### 3. Phantom citations

How many fake citations were caught? ___
How would you have discovered this in production, if the check did not exist?

### 4. The threshold you would ship

Threshold: ___  At that setting it refuses ___/20 answerable and ___/5 unanswerable.

Why is there no setting that catches all five without refusing real questions?

### 5. The two answers the code read wrongly

Pick A15 or U01 from Step 5.

Question id: ___  What did the model do, what did the service decide, and what code change fixes it?

### 5b. Citation precision and citation recall

Cites only what it used (citation precision): ___   Every claim cited (citation recall): ___

Which one would you add to the service as a check that refuses, and which would you only report? Why?

### 6. What you would fix first, and how you would prove it worked